# E-Commerce Data Cleaning and Analysis

This notebook performs comprehensive data cleaning and analysis on the Brazilian E-Commerce dataset from Kaggle (Olist marketplace). The dataset includes order transactions, customer information, product details, reviews, payments, sellers, and geolocation data.

## Step 1: Setup and Import Libraries

Import necessary libraries for data manipulation, file handling, and downloading datasets from Kaggle.

In [1]:
# Import required libraries
import kagglehub
import os
import pandas as pd

/Users/hilal/Documents/Python Projects/Data Analysis/E-Commerce Project/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Function to analyze duplicates

In [2]:
def analyze_duplicates(datasets):
    # Analyze duplicates in key identifier columns for each dataset
    # This helps identify data quality issues and potential errors
    print("=" * 70)
    print("DUPLICATE ANALYSIS FOR EACH DATASET")
    print("=" * 70)

    # Map each dataset to its primary key column for duplicate checking
    key_columns = {
        "olist_orders_dataset": "order_id",
        "olist_customers_dataset": "customer_id",
        "olist_products_dataset": "product_id",
        "olist_order_reviews_dataset": "review_id",
        "olist_sellers_dataset": "seller_id",
        "olist_geolocation_dataset": "geolocation_zip_code_prefix",
        "product_category_name_translation": "product_category_name"
    }

    # Initialize dictionary to store duplicate analysis results
    duplicate_summary = {}

    excluded_datasets = ["olist_order_items_dataset", "olist_order_payments_dataset", "olist_geolocation_dataset"]

    # Iterate through each dataset and analyze duplicates
    for dataset_name, df in datasets.items():
        if dataset_name in excluded_datasets:
            continue

        print(f"\n📊 Dataset: {dataset_name}")
        print(f"   Shape: {df.shape}")
        
        # Retrieve the primary key column for this dataset
        key_col = key_columns.get(dataset_name)
        
        # Check if key column exists in the dataset
        if key_col and key_col in df.columns:
            # Calculate total rows and unique values in the key column
            total_rows = len(df)
            unique_count = df[key_col].nunique()
            duplicate_count = total_rows - unique_count
            
            # Find all duplicate values and their frequencies
            duplicates = df[df.duplicated(subset=[key_col], keep=False)][key_col].value_counts()
            
            # Store the analysis results in a summary dictionary
            duplicate_summary[dataset_name] = {
                "key_column": key_col,
                "total_rows": total_rows,
                "unique_values": unique_count,
                "duplicate_rows": duplicate_count,
                "duplicate_percentage": (duplicate_count / total_rows * 100) if total_rows > 0 else 0
            }
            
            # Print detailed duplicate analysis for this dataset
            print(f"   Key Column: {key_col}")
            print(f"   Total Rows: {total_rows}")
            print(f"   Unique Values: {unique_count}")
            print(f"   Duplicate Rows: {duplicate_count} ({duplicate_summary[dataset_name]['duplicate_percentage']:.2f}%)")
            
            # Display top 5 duplicate values if any exist
            if duplicate_count > 0:
                print(f"   Top Duplicates in '{key_col}':")
                print(f"   {duplicates.head(5).to_dict()}")
        else:
            # Handle case where key column doesn't exist in dataset
            print(f"   ⚠️  Could not find key column '{key_col}' in dataset")
            print(f"   Available columns: {list(df.columns)}")

    # Create a summary table of all duplicate analysis results
    print("\n" + "=" * 70)
    print("SUMMARY TABLE")
    print("=" * 70)
    summary_df = pd.DataFrame(duplicate_summary).T
    print(summary_df)

## Step 2: Download and Explore Dataset

Download the Brazilian E-Commerce dataset from Kaggle using the kagglehub API and explore the available files.

In [3]:
# Download the Brazilian E-Commerce dataset from Kaggle
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

Path to dataset files: /Users/hilal/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2


### Dataset Path and Files

Display the path where the dataset is stored and list all available CSV files.

In [4]:
# List all files in the downloaded dataset directory
files = os.listdir(path)

# Print each file name to see what datasets are available
print("Files in dataset:")
for file in files:
    print(file)

Files in dataset:
olist_sellers_dataset.csv
product_category_name_translation.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv


## Step 3: Load and Store Datasets

Load all CSV files into a dictionary of pandas DataFrames for easy access and manipulation.

In [5]:
# Filter and extract only CSV files from the dataset directory
csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]

# Initialize a dictionary to store all datasets
datasets = {}

# Load each CSV file into a pandas DataFrame and store in the dictionary
for file in csv_files:
    name = file.replace(".csv", "")  # Remove .csv extension to use as key
    datasets[name] = pd.read_csv(os.path.join(path, file))

# Display all loaded dataset names
print(datasets.keys())

dict_keys(['olist_sellers_dataset', 'product_category_name_translation', 'olist_orders_dataset', 'olist_order_items_dataset', 'olist_customers_dataset', 'olist_geolocation_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset', 'olist_products_dataset'])


## Step 4: Explore Data Structure

Display all column names for each dataset to understand the data structure and available fields.

In [6]:
# Iterate through all datasets and display their column names
for name, df in datasets.items():
    print(f"\nDataset: {name}")
    # Print all column names for each dataset to understand data structure
    print(df.columns.tolist())


Dataset: olist_sellers_dataset
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

Dataset: product_category_name_translation
['product_category_name', 'product_category_name_english']

Dataset: olist_orders_dataset
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Dataset: olist_order_items_dataset
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Dataset: olist_customers_dataset
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Dataset: olist_geolocation_dataset
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Dataset: olist_order_payments_dataset
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Dataset: olist_o

## Step 5: Data Cleaning

Fix column naming inconsistencies and prepare data for analysis. Rename misspelled column names in the products dataset for better readability.

In [7]:
#Renaming the columns in olist_products_dataset for better readability
datasets["olist_products_dataset"].rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
}, inplace=True)

datasets["olist_products_dataset"].columns

Index(['product_id', 'product_category_name', 'product_name_length',
       'product_description_length', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')

### Analyze Duplicates

Analyze duplicate records in key identifier columns across all datasets. This helps identify data integrity issues and potential errors that need to be addressed during the cleaning process.

### Key Metrics:
- **Total Rows**: Number of records in the dataset
- **Unique Values**: Count of distinct key identifiers
- **Duplicate Rows**: Number of records with duplicate identifiers
- **Duplicate Percentage**: Percentage of data that contains duplicates

In [8]:
#Checking for duplicates in key identifier columns across all datasets
analyze_duplicates(datasets)

DUPLICATE ANALYSIS FOR EACH DATASET

📊 Dataset: olist_sellers_dataset
   Shape: (3095, 4)
   Key Column: seller_id
   Total Rows: 3095
   Unique Values: 3095
   Duplicate Rows: 0 (0.00%)

📊 Dataset: product_category_name_translation
   Shape: (71, 2)
   Key Column: product_category_name
   Total Rows: 71
   Unique Values: 71
   Duplicate Rows: 0 (0.00%)

📊 Dataset: olist_orders_dataset
   Shape: (99441, 8)
   Key Column: order_id
   Total Rows: 99441
   Unique Values: 99441
   Duplicate Rows: 0 (0.00%)

📊 Dataset: olist_customers_dataset
   Shape: (99441, 5)
   Key Column: customer_id
   Total Rows: 99441
   Unique Values: 99441
   Duplicate Rows: 0 (0.00%)

📊 Dataset: olist_order_reviews_dataset
   Shape: (99224, 7)
   Key Column: review_id
   Total Rows: 99224
   Unique Values: 98410
   Duplicate Rows: 814 (0.82%)
   Top Duplicates in 'review_id':
   {'c444278834184f72b1484dfe47de7f97': 3, '308316408775d1600dad81bd3184556d': 3, '2d6ac45f859465b5c185274a1c929637': 3, '3415c9f764e47840

### Remove Duplicate Reviews

Remove duplicate review records based on the review_id column to ensure data integrity.

In [9]:
# Remove rows that are completely identical
datasets["olist_order_reviews_dataset"] = datasets["olist_order_reviews_dataset"].drop_duplicates(subset=['review_id'], keep='first')

In [10]:
#Checking for duplicates in key identifier columns across all datasets
analyze_duplicates(datasets)

DUPLICATE ANALYSIS FOR EACH DATASET

📊 Dataset: olist_sellers_dataset
   Shape: (3095, 4)
   Key Column: seller_id
   Total Rows: 3095
   Unique Values: 3095
   Duplicate Rows: 0 (0.00%)

📊 Dataset: product_category_name_translation
   Shape: (71, 2)
   Key Column: product_category_name
   Total Rows: 71
   Unique Values: 71
   Duplicate Rows: 0 (0.00%)

📊 Dataset: olist_orders_dataset
   Shape: (99441, 8)
   Key Column: order_id
   Total Rows: 99441
   Unique Values: 99441
   Duplicate Rows: 0 (0.00%)

📊 Dataset: olist_customers_dataset
   Shape: (99441, 5)
   Key Column: customer_id
   Total Rows: 99441
   Unique Values: 99441
   Duplicate Rows: 0 (0.00%)

📊 Dataset: olist_order_reviews_dataset
   Shape: (98410, 7)
   Key Column: review_id
   Total Rows: 98410
   Unique Values: 98410
   Duplicate Rows: 0 (0.00%)

📊 Dataset: olist_products_dataset
   Shape: (32951, 9)
   Key Column: product_id
   Total Rows: 32951
   Unique Values: 32951
   Duplicate Rows: 0 (0.00%)

SUMMARY TABLE
   

### Check for Null Values

Analyze missing values (NaN/NULL) across all columns in each dataset to identify data quality issues.

In [11]:
# Check for null values in all datasets
print("=" * 70)
print("NULL VALUES ANALYSIS")
print("=" * 70)

for dataset_name, df in datasets.items():
    null_counts = df.isnull().sum()
    null_percent = (df.isnull().sum() / len(df)) * 100
    
    print(f"\n📊 Dataset: {dataset_name}")
    print(f"   Shape: {df.shape}")
    
    if null_counts.sum() > 0:
        print(f"   Columns with NULL values:")
        for col in null_counts[null_counts > 0].index:
            print(f"      - {col}: {null_counts[col]} ({null_percent[col]:.2f}%)")
    else:
        print(f"   ✅ No null values found")

NULL VALUES ANALYSIS

📊 Dataset: olist_sellers_dataset
   Shape: (3095, 4)
   ✅ No null values found

📊 Dataset: product_category_name_translation
   Shape: (71, 2)
   ✅ No null values found

📊 Dataset: olist_orders_dataset
   Shape: (99441, 8)
   Columns with NULL values:
      - order_approved_at: 160 (0.16%)
      - order_delivered_carrier_date: 1783 (1.79%)
      - order_delivered_customer_date: 2965 (2.98%)

📊 Dataset: olist_order_items_dataset
   Shape: (112650, 7)
   ✅ No null values found

📊 Dataset: olist_customers_dataset
   Shape: (99441, 5)
   ✅ No null values found

📊 Dataset: olist_geolocation_dataset
   Shape: (1000163, 5)
   ✅ No null values found

📊 Dataset: olist_order_payments_dataset
   Shape: (103886, 5)
   ✅ No null values found

📊 Dataset: olist_order_reviews_dataset
   Shape: (98410, 7)
   Columns with NULL values:
      - review_comment_title: 86891 (88.29%)
      - review_comment_message: 57742 (58.67%)

📊 Dataset: olist_products_dataset
   Shape: (32951, 9)
 

### Clean Inconsistent Order Data

Remove orders with inconsistent delivery status and dates. An order marked as 'delivered' must have delivery date records.

In [12]:
# Count rows before deletion
before_count = len(datasets["olist_orders_dataset"])

# Remove rows where order_approved_at is null AND order_status is 'delivered'
mask = (datasets["olist_orders_dataset"]["order_approved_at"].isnull()) & (datasets["olist_orders_dataset"]["order_status"] == 'delivered')
datasets["olist_orders_dataset"] = datasets["olist_orders_dataset"][~mask]

# Count rows after deletion
after_count = len(datasets["olist_orders_dataset"])

print(f"Rows removed: {before_count - after_count}")
print(f"Rows before: {before_count}")
print(f"Rows after: {after_count}")

Rows removed: 14
Rows before: 99441
Rows after: 99427


In [13]:
# Count rows before deletion
before_count = len(datasets["olist_orders_dataset"])

# Remove rows where order_delivered_carrier_date is null AND order_status is 'delivered'
mask = (datasets["olist_orders_dataset"]["order_delivered_carrier_date"].isnull()) & (datasets["olist_orders_dataset"]["order_status"] == 'delivered')
datasets["olist_orders_dataset"] = datasets["olist_orders_dataset"][~mask]

# Count rows after deletion
after_count = len(datasets["olist_orders_dataset"])

print(f"Rows removed: {before_count - after_count}")
print(f"Rows before: {before_count}")
print(f"Rows after: {after_count}")

Rows removed: 2
Rows before: 99427
Rows after: 99425


In [14]:
# Count rows before deletion
before_count = len(datasets["olist_orders_dataset"])

# Remove rows where order_delivered_customer_date is null AND order_status is 'delivered'
mask = (datasets["olist_orders_dataset"]["order_delivered_customer_date"].isnull()) & (datasets["olist_orders_dataset"]["order_status"] == 'delivered')
datasets["olist_orders_dataset"] = datasets["olist_orders_dataset"][~mask]

# Count rows after deletion
after_count = len(datasets["olist_orders_dataset"])

print(f"Rows removed: {before_count - after_count}")
print(f"Rows before: {before_count}")
print(f"Rows after: {after_count}")

Rows removed: 7
Rows before: 99425
Rows after: 99418


### Validate Price and Payment Values

Check for invalid prices and payment amounts (negative values, zeros, and nulls) in order items and payment datasets.

In [15]:
# Comprehensive invalid data check
print("=" * 70)
print("INVALID PRICE CHECK")
print("=" * 70)

invalid_summary = {}

# Check items dataset
df_items = datasets["olist_order_items_dataset"]
invalid_summary["Order Items - Negative prices"] = (df_items["price"] < 0).sum()
invalid_summary["Order Items - Zero prices"] = (df_items["price"] == 0).sum()
invalid_summary["Order Items - Null prices"] = df_items["price"].isnull().sum()
invalid_summary["Order Items - Negative freight"] = (df_items["freight_value"] < 0).sum()

# Check payments dataset
df_payments = datasets["olist_order_payments_dataset"]
invalid_summary["Order Payments - Negative values"] = (df_payments["payment_value"] < 0).sum()
invalid_summary["Order Payments - Zero values"] = (df_payments["payment_value"] == 0).sum()
invalid_summary["Order Payments - Null values"] = df_payments["payment_value"].isnull().sum()

# Display results
for check, count in invalid_summary.items():
    status = "❌" if count > 0 else "✅"
    print(f"{status} {check}: {count}")

INVALID PRICE CHECK
✅ Order Items - Negative prices: 0
✅ Order Items - Zero prices: 0
✅ Order Items - Null prices: 0
✅ Order Items - Negative freight: 0
✅ Order Payments - Negative values: 0
❌ Order Payments - Zero values: 9
✅ Order Payments - Null values: 0


### Analyze Data Types

Review the data types of all columns in each dataset to identify type inconsistencies and conversion needs.

In [16]:
# Check data types for all datasets
print("=" * 70)
print("DATA TYPES ANALYSIS")
print("=" * 70)

for dataset_name, df in datasets.items():
    print(f"\n📊 Dataset: {dataset_name}")
    print(df.dtypes)

DATA TYPES ANALYSIS

📊 Dataset: olist_sellers_dataset
seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object

📊 Dataset: product_category_name_translation
product_category_name            str
product_category_name_english    str
dtype: object

📊 Dataset: olist_orders_dataset
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

📊 Dataset: olist_order_items_dataset
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

📊 Dataset: olist_customers_dataset
customer_id           

## Step 6: Identify and Convert Date/Timestamp Columns

Identify all date and timestamp columns across datasets and prepare them for conversion to proper datetime format.

In [17]:
# Create a comprehensive list of all date and timestamp columns across datasets
date_cols = [
    # olist_orders_dataset
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    # olist_order_items_dataset
    "shipping_limit_date",
    # olist_order_reviews_dataset
    "review_creation_date",
    "review_answer_timestamp"
]

# Verify which date columns exist in the datasets and display them
print("=" * 70)
print("DATE AND TIMESTAMP COLUMNS MAPPING")
print("=" * 70)

found_date_cols = {}

for dataset_name, df in datasets.items():
    dataset_date_cols = [col for col in date_cols if col in df.columns]
    if dataset_date_cols:
        found_date_cols[dataset_name] = dataset_date_cols
        print(f"\n📅 {dataset_name}:")
        for col in dataset_date_cols:
            print(f"   - {col}: {df[col].dtype}")

print(f"\n\n✅ Total date/timestamp columns found: {sum(len(v) for v in found_date_cols.values())}")
print(f"📊 Datasets containing date columns: {len(found_date_cols)}")


DATE AND TIMESTAMP COLUMNS MAPPING

📅 olist_orders_dataset:
   - order_purchase_timestamp: str
   - order_approved_at: str
   - order_delivered_carrier_date: str
   - order_delivered_customer_date: str
   - order_estimated_delivery_date: str

📅 olist_order_items_dataset:
   - shipping_limit_date: str

📅 olist_order_reviews_dataset:
   - review_creation_date: str
   - review_answer_timestamp: str


✅ Total date/timestamp columns found: 8
📊 Datasets containing date columns: 3


### Convert Date Columns to Datetime Format

Transform all identified date and timestamp columns to proper datetime64[ns] format for time-series analysis and date operations.

In [18]:
# Convert all date and timestamp columns to datetime64[us] format
print("=" * 70)
print("CONVERTING DATE/TIMESTAMP COLUMNS TO DATETIME64[us]")
print("=" * 70)

conversion_summary = {}

# Iterate through each dataset and convert date columns
for dataset_name, df in datasets.items():
    # Get date columns that exist in this dataset
    dataset_date_cols = [col for col in date_cols if col in df.columns]
    
    if dataset_date_cols:
        print(f"\n📅 {dataset_name}:")
        conversion_summary[dataset_name] = {}
        
        for col in dataset_date_cols:
            # Store original dtype
            original_dtype = df[col].dtype
            
            try:
                # Convert to datetime64[us]
                datasets[dataset_name][col] = pd.to_datetime(df[col], errors='coerce')
                new_dtype = datasets[dataset_name][col].dtype
                
                # Count null values introduced by conversion errors
                null_count = datasets[dataset_name][col].isnull().sum()
                
                conversion_summary[dataset_name][col] = {
                    "original_dtype": str(original_dtype),
                    "new_dtype": str(new_dtype),
                    "null_values": null_count
                }
                
                print(f"   ✅ {col}")
                print(f"      {original_dtype} → {new_dtype}")
                if null_count > 0:
                    print(f"      ⚠️  {null_count} values could not be converted (set to NaT)")
            except Exception as e:
                print(f"   ❌ {col}: {str(e)}")

# Display conversion summary
print("\n" + "=" * 70)
print("CONVERSION SUMMARY")
print("=" * 70)

for dataset_name, conversions in conversion_summary.items():
    print(f"\n{dataset_name}:")
    for col, info in conversions.items():
        print(f"   {col}: {info['original_dtype']} → {info['new_dtype']} ({info['null_values']} nulls)")

print("\n✅ All date/timestamp columns converted to datetime64[us]!")

CONVERTING DATE/TIMESTAMP COLUMNS TO DATETIME64[us]

📅 olist_orders_dataset:
   ✅ order_purchase_timestamp
      str → datetime64[us]
   ✅ order_approved_at
      str → datetime64[us]
      ⚠️  146 values could not be converted (set to NaT)
   ✅ order_delivered_carrier_date
      str → datetime64[us]
      ⚠️  1781 values could not be converted (set to NaT)
   ✅ order_delivered_customer_date
      str → datetime64[us]
      ⚠️  2957 values could not be converted (set to NaT)
   ✅ order_estimated_delivery_date
      str → datetime64[us]

📅 olist_order_items_dataset:
   ✅ shipping_limit_date
      str → datetime64[us]

📅 olist_order_reviews_dataset:
   ✅ review_creation_date
      str → datetime64[us]
   ✅ review_answer_timestamp
      str → datetime64[us]

CONVERSION SUMMARY

olist_orders_dataset:
   order_purchase_timestamp: str → datetime64[us] (0 nulls)
   order_approved_at: str → datetime64[us] (146 nulls)
   order_delivered_carrier_date: str → datetime64[us] (1781 nulls)
   order_d

### Verify Converted Data Types

Display updated data types for all columns after datetime conversion to confirm the changes were applied successfully.

In [19]:
# Check data types for all datasets
print("=" * 70)
print("DATA TYPES ANALYSIS")
print("=" * 70)

for dataset_name, df in datasets.items():
    print(f"\n📊 Dataset: {dataset_name}")
    print(df.dtypes)

DATA TYPES ANALYSIS

📊 Dataset: olist_sellers_dataset
seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object

📊 Dataset: product_category_name_translation
product_category_name            str
product_category_name_english    str
dtype: object

📊 Dataset: olist_orders_dataset
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

📊 Dataset: olist_order_items_dataset
order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price      